# Generalized LLM Grading Pipeline — Anchored & Original (OpenAI / Anthropic / Gemini)


Select a JSON **with answers**, choose **LLM provider/model**, pick **grading schema** (`anchored` or `original`),
watch a **progress counter**, and get a **flat JSON** of results (each item contains an `evaluation` object with
`scores`, `total`, `comments`, and `rationales`).


In [ ]:
# 1) Config — paths and checkpoint
from pathlib import Path

# Input: answered JSON (each record should include question + model_answer)
ANSWERS_PATH = Path('../examples/sample_answered.json')   # <-- change as needed

# Output: flat JSON list with `evaluation` injected into each item
OUT_PATH     = Path('outputs/graded_sample.json')                  # <-- change as needed

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Save checkpoints every N items (0 to disable)
CHECKPOINT_EVERY = 0


In [ ]:
import os

# 2) API keys — read from the environment (or a .env file). Never hardcode a key here.
#    export OPENAI_API_KEY=... / ANTHROPIC_API_KEY=... / GOOGLE_API_KEY=...

# Load a .env from the repo root if one exists (stdlib only, no extra dependency).
def _load_dotenv(*candidates):
    for p in candidates:
        if p.exists():
            for line in p.read_text().splitlines():
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    k, v = line.split("=", 1)
                    os.environ.setdefault(k.strip(), v.strip().strip("'\""))
            return p
    return None

_load_dotenv(Path(".env"), Path("../.env"))

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")
GEMINI_API_KEY    = os.getenv("GOOGLE_API_KEY", "")


In [ ]:
# 3) Choices — provider, model, schema
# Provider options: 'openai', 'anthropic', 'gemini'
PROVIDER = 'openai'

# Model for chosen provider
GRADER_MODEL = "gpt-4o-mini"             # OpenAI example
# GRADER_MODEL = "claude-3-5-sonnet-latest"  # Anthropic example
# GRADER_MODEL = "gemini-1.5-pro"            # Gemini example

# Schema mode: 'original' OR 'anchored'
# - 'original': score each category independently on a 0.00-5.00 scale
# - 'anchored': start every category at 2.50 and adjust up or down from there
SCHEMA_MODE = 'original'  # or 'anchored'


In [ ]:
# 4) Consistent system writer + grading guides + schema instructions
SYSTEM_GRADER = (
    "You are a strict, reproducible grader."
    "Do not include chain-of-thought. Do not include any text outside the final JSON."
    "Follow the grading guide and schema instructions exactly."
)

# --- ORIGINAL (unanchored) GRADING GUIDE ---
GRADING_GUIDE_ORIGINAL = """
Use the following rubric with seven categories. For each category, assign a score within
the range [0.00, 5.00] that reflects the quality of the response along that dimension.
Round to TWO decimals. Make sure all steps are in increments of 0.1, so scores for each category like 3.3, 3.2, 4.7 can exist.

Categories:
- relevance_task
- factual_accuracy
- coherence_structure
- depth_insight
- linguistic_quality
- instruction_sensitivity
- creativity_originality

Rules:
1) Judge each category independently.
2) No chain-of-thought in output; just JSON verdicts.
3) Two decimals for all numbers.
""".strip()

# --- ANCHORED GRADING GUIDE ---
GRADING_GUIDE_ANCHORED = """
Use an ANCHORED rubric with seven categories. Start each category at 2.50 on a 0.00–5.00 scale.
Adjust up or down based on evidence, then clamp to [0.00, 5.00]. Round to TWO decimals.

Categories:
- relevance_task
- factual_accuracy
- coherence_structure
- depth_insight
- linguistic_quality
- instruction_sensitivity
- creativity_originality

Balanced adjustments (examples):
- Minor issue/merit: ±0.25 to ±0.50
- Moderate: ±0.75 to ±1.25
- Major OR Exceptional: ±1.50 to ±2.00

Rules:
1) Judge each category independently.
2) No chain-of-thought in output; just JSON verdicts.
3) Two decimals for all numbers.
""".strip()

# --- SCHEMA INSTRUCTIONS (shared) ---
SCHEMA_INSTRUCTIONS = """
Return ONLY valid JSON:
{
  "scores": {
    "relevance_task": "0.00–5.00",
    "factual_accuracy": "0.00–5.00",
    "coherence_structure": "0.00–5.00",
    "depth_insight": "0.00–5.00",
    "linguistic_quality": "0.00–5.00",
    "instruction_sensitivity": "0.00–5.00",
    "creativity_originality": "0.00–5.00"
  },
  "total": "sum of seven (two decimals)",
  "comments": "one-sentence rationale",
  "rationales": { "relevance_task": "short reason", "...": "..." }
}
""".strip()

SCORING_DIMENSIONS = [
    "relevance_task",
    "factual_accuracy",
    "coherence_structure",
    "depth_insight",
    "linguistic_quality",
    "instruction_sensitivity",
    "creativity_originality",
]


In [ ]:
# 5) I/O helpers — robust reader and flat writer
import json
from json import JSONDecodeError
from typing import Any, Dict, List

def read_json_like(path: Path):
    txt = path.read_text(encoding="utf-8-sig")
    try:
        return json.loads(txt)
    except JSONDecodeError:
        data = []
        for ln in txt.splitlines():
            s = ln.strip()
            if s:
                data.append(json.loads(s))
        return data

def to_list(data: Any) -> List[Dict[str, Any]]:
    if isinstance(data, list):
        return [dict(x) if isinstance(x, dict) else {"question": str(x)} for x in data]
    if isinstance(data, dict):
        for k in ("outputs", "items", "data"):
            if isinstance(data.get(k), list):
                return [dict(x) if isinstance(x, dict) else {"question": str(x)} for x in data[k]]
        return [dict(data)]
    return [{"question": str(data)}]

def get_question(rec: Dict[str, Any]) -> str:
    for k in ("question","prompt","text","instruction","input","content"):
        v = rec.get(k)
        if isinstance(v, str) and v.strip():
            return v.strip()
    return ""

def get_model_answer(rec: Dict[str, Any]) -> str:
    return (rec.get("model_answer") or rec.get("answer") or rec.get("response") or "").strip()

def get_reference(rec: Dict[str, Any]) -> str:
    for k in ("reference","reference_answer","gold","target","ground_truth","expected","ideal"):
        v = rec.get(k)
        if isinstance(v, str) and v.strip():
            return v.strip()
    return ""

def flat_write(path: Path, rows: List[Dict[str, Any]]):
    path.write_text(json.dumps(rows, indent=2, ensure_ascii=False), encoding="utf-8")


In [ ]:
# 6) Providers — OpenAI / Anthropic / Gemini (deterministic caller params)
import time, random
from typing import Optional

MAX_RETRIES = 5
BASE_BACKOFF = 1.5

def _backoff_sleep(attempt: int):
    delay = (BASE_BACKOFF ** attempt) + random.random()
    time.sleep(min(delay, 30.0))

class ProviderError(RuntimeError): pass

class BaseProvider:
    name: str = "base"
    def available(self) -> bool: raise NotImplementedError
    def generate_json(self, system: str, user: str, *, temperature: float=0.0, max_tokens: int=700) -> str: raise NotImplementedError

class OpenAIProvider(BaseProvider):
    name = "openai"
    def __init__(self, model="gpt-4o-mini", api_key: Optional[str]=None):
        self.model = model
        self.api_key = api_key
        try:
            from openai import OpenAI
            self._client = OpenAI(api_key=self.api_key) if self.api_key else None
        except Exception:
            self._client = None
    def available(self) -> bool: return self._client is not None
    def generate_json(self, system: str, user: str, *, temperature: float=0.0, max_tokens: int=700) -> str:
        if not self.available():
            raise ProviderError("OpenAI not available — provide OPENAI_API_KEY and install openai.")
        last_err = None
        for attempt in range(1, MAX_RETRIES+1):
            try:
                resp = self._client.chat.completions.create(
                    model=self.model,
                    messages=[
                        {"role":"system","content": system},
                        {"role":"user","content": user},
                    ],
                    temperature=0.0, top_p=1, max_tokens=max_tokens
                )
                return (resp.choices[0].message.content or "").strip()
            except Exception as e:
                last_err = e
                _backoff_sleep(attempt)
        raise ProviderError(f"OpenAI failed after retries: {last_err!r}")

class AnthropicProvider(BaseProvider):
    name = "anthropic"
    def __init__(self, model="claude-3-5-sonnet-latest", api_key: Optional[str]=None):
        self.model = model
        self.api_key = api_key
        try:
            import anthropic
            self._client = anthropic.Anthropic(api_key=self.api_key) if self.api_key else None
        except Exception:
            self._client = None
    def available(self) -> bool: return self._client is not None
    def generate_json(self, system: str, user: str, *, temperature: float=0.0, max_tokens: int=700) -> str:
        if not self.available():
            raise ProviderError("Anthropic not available — provide ANTHROPIC_API_KEY and install anthropic.")
        last_err = None
        for attempt in range(1, MAX_RETRIES+1):
            try:
                msg = self._client.messages.create(
                    model=self.model, system=system,
                    messages=[{"role":"user","content": user}],
                    temperature=0.0, max_tokens=max_tokens
                )
                parts = getattr(msg, "content", [])
                return "".join([p.text for p in parts if hasattr(p,"text")]).strip()
            except Exception as e:
                last_err = e
                _backoff_sleep(attempt)
        raise ProviderError(f"Anthropic failed after retries: {last_err!r}")

class GeminiProvider(BaseProvider):
    name = "gemini"
    def __init__(self, model="gemini-1.5-pro", api_key: Optional[str]=None):
        self.model = model
        self.api_key = api_key
        try:
            import google.generativeai as genai
            if self.api_key:
                genai.configure(api_key=self.api_key)
                try:
                    self._model = genai.GenerativeModel(self.model, system_instruction=SYSTEM_GRADER)
                except TypeError:
                    self._model = genai.GenerativeModel(self.model)
            else:
                self._model = None
        except Exception:
            self._model = None
    def available(self) -> bool: return self._model is not None
    def generate_json(self, system: str, user: str, *, temperature: float=0.0, max_tokens: int=700) -> str:
        if not self.available():
            raise ProviderError("Gemini not available — provide GEMINI_API_KEY and install google-generativeai.")
        last_err = None
        for attempt in range(1, MAX_RETRIES+1):
            try:
                content = user
                try:
                    _ = self._model.generate_content
                except Exception:
                    content = system + "\n\n" + user
                resp = self._model.generate_content(
                    content,
                    generation_config={"temperature":0.0, "top_p":1, "max_output_tokens":max_tokens}
                )
                txt = getattr(resp, "text", None)
                if txt is None:
                    cand = (getattr(resp, "candidates", None) or [{}])[0]
                    txt = cand.get("content", {}).get("parts", [{}])[0].get("text", "")
                return (txt or "").strip()
            except Exception as e:
                last_err = e
                _backoff_sleep(attempt)
        raise ProviderError(f"Gemini failed after retries: {last_err!r}")

def get_provider(name: str, model: str):
    k = (name or "").lower()
    if k == "openai":   return OpenAIProvider(model=model, api_key=(OPENAI_API_KEY or None))
    if k == "anthropic":return AnthropicProvider(model=model, api_key=(ANTHROPIC_API_KEY or None))
    if k == "gemini":   return GeminiProvider(model=model, api_key=(GEMINI_API_KEY or None))
    raise ValueError(f"Unknown provider: {name}")


In [ ]:
# 7) Grading loop — progress counter + flat JSON output with `evaluation`
import json, re, datetime as dt

def extract_json(text: str) -> dict:
    """Parse direct JSON or extract first {...} block."""
    if not text:
        return {}
    try:
        return json.loads(text)
    except Exception:
        pass
    m = re.search(r'\{.*\}', text, flags=re.S)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            return {}
    return {}

def round2(x):
    try:
        return round(float(x), 2)
    except Exception:
        return 0.0

def coerce_to_evaluation(obj: dict) -> dict:
    """
    Normalize grader output into:
    {
      "scores": {... seven keys (0.00..5.00, 2 decimals) ...},
      "total": float(two decimals),
      "comments": str,
      "rationales": { ... optional ... }
    }
    """
    scores_in  = (obj.get("scores") or obj.get("evaluation", {}).get("scores") or {})
    comments   = obj.get("comments") or obj.get("evaluation", {}).get("comments") or ""
    rationales = obj.get("rationales") or obj.get("evaluation", {}).get("rationales") or {}

    dims = [
        "relevance_task",
        "factual_accuracy",
        "coherence_structure",
        "depth_insight",
        "linguistic_quality",
        "instruction_sensitivity",
        "creativity_originality",
    ]
    scores = {k: round2(scores_in.get(k, 0.0)) for k in dims}
    total = round(sum(scores.values()), 2)

    return {
        "scores": scores,
        "total": total,
        "comments": comments,
        "rationales": rationales
    }

def build_user_prompt(schema_mode: str, question: str, model_answer: str, reference_answer: str) -> str:
    guide = GRADING_GUIDE_ANCHORED if schema_mode == "anchored" else GRADING_GUIDE_ORIGINAL
    if schema_mode == "anchored":
        return (
            f"{guide}\n\n"
            f"{SCHEMA_INSTRUCTIONS}\n\n"
            f"QUESTION:\n{question}\n\n"
            f"MODEL_ANSWER:\n{model_answer}\n\n"
            f"REFERENCE_ANSWER (may be empty):\n{reference_answer}\n\n"
            "Return the JSON object now."
        )
    else:
        return (
            f"{guide}\n\n"
            f"{SCHEMA_INSTRUCTIONS}\n\n"
            f"QUESTION:\n{question}\n\n"
            f"MODEL_ANSWER:\n{model_answer}\n\n"
            "Return the JSON object now."
        )

prov = get_provider(PROVIDER, GRADER_MODEL)
if not prov.available():
    raise RuntimeError(f"Provider '{PROVIDER}' not available — check API key/SDK.")

raw = read_json_like(ANSWERS_PATH)
items = to_list(raw)
N = len(items)

graded = []

for i, rec in enumerate(items, start=1):
    q   = get_question(rec)
    a   = get_model_answer(rec)
    ref = get_reference(rec)
    pid = rec.get("id") or rec.get("name") or rec.get("topic") or f"index_{i}"

    print(f"[{i}/{N}] Grading: {pid}")

    user_prompt = build_user_prompt(SCHEMA_MODE, q, a, ref)

    try:
        txt = prov.generate_json(SYSTEM_GRADER, user_prompt, temperature=0.0, max_tokens=700)
        parsed = extract_json(txt)
        evaluation = coerce_to_evaluation(parsed)
    except Exception as e:
        evaluation = {
            "scores": {
                "relevance_task": 0.0, "factual_accuracy": 0.0, "coherence_structure": 0.0,
                "depth_insight": 0.0, "linguistic_quality": 0.0, "instruction_sensitivity": 0.0,
                "creativity_originality": 0.0
            },
            "total": 0.0,
            "comments": f"[ERROR] {e}",
            "rationales": {}
        }

    out_rec = dict(rec)
    out_rec["evaluation"] = evaluation
    graded.append(out_rec)

    # checkpoint
    if CHECKPOINT_EVERY and (i % CHECKPOINT_EVERY == 0):
        ck = OUT_PATH.with_name(OUT_PATH.stem + f"__checkpoint_{i}.json")
        ck.write_text(json.dumps(graded, indent=2, ensure_ascii=False), encoding="utf-8")

# final save (flat JSON list)
OUT_PATH.write_text(json.dumps(graded, indent=2, ensure_ascii=False), encoding="utf-8")
print("\nSaved graded results (flat JSON):", str(OUT_PATH))
